# Phase 2 — Modern datasets & the cross-dataset generalization grid

This notebook builds **directly on the Phase-1 core above** (leak-free splits, silence-trim
preprocessing, CNN-LSTM + CLIP models, EER/AUC/CI metrics, seeded training & eval). Phase 2 adds
the two deliverables from §8 of the plan:

1. **Per-dataset loaders** that normalise every corpus to one `(path, label, meta)` schema with
   **provenance** fields (source URL · license · version · access date), and
2. the **cross-dataset generalization grid** — *train on X, test on Y for all pairs* — which is the
   core paper figure, plus **leave-one-generator-out (LOGO)** scaffolding.

### Datasets (all stats carry a `VERIFY:` gate — confirm against the primary source before citing)

| Dataset | HF / source id | Role | License (VERIFY) | Notes |
|---|---|---|---|---|
| **WaveFake** | `ajaykarthick/wavefake-audio` | legacy in-dist | VERIFY | from Phase 1 |
| **CodecFake** | `ajaykarthick/codecfake-audio` | neural-codec artifacts | CC BY-NC-ND 4.0 (Xie et al.) / VCTK CC-BY-4.0 (Tseng) — VERIFY which variant | same uploader as WaveFake → likely same schema (tolerant loader) |
| **In-the-Wild** | `mueller91/In-The-Wild` | real-world generalization stress test | research-only — VERIFY | ~58 speakers, ≈20.8h bona-fide / 17.2h spoof; **speaker-disjoint** splits |
| **MLAAD** | `mueller91/MLAAD` | cross-generator / cross-lingual | **CC-BY-NC 4.0** (v8+) | spoof-only (140+ TTS, 51+ langs); real from **M-AILABS**; carries generator+language for LOGO |
| **ASVspoof 2019 LA** | official protocol files | metric-rich in-domain | research-only — VERIFY | `system_id` (A01–A19) → LOGO; **min-tDCF** |
| **ASVspoof 5** | official challenge release | modern benchmark | research-only — VERIFY | minDCF/EER + **a-DCF** via the official eval package |
| **FoR** | Kaggle | legacy | VERIFY | from Phase 1 |

> **Honesty note (carried from the plan).** Web/HF access was blocked while authoring this, so the
> ids/licenses above come from search results and **must be re-verified on the dataset card** before
> any number reaches a paper. Every loader records provenance with a `verified=False` flag you flip
> once you've checked the primary source.

**Runnability:** HF loaders (WaveFake/CodecFake/In-the-Wild) run on Colab with no token. MLAAD,
ASVspoof 2019 LA and ASVspoof 5 are **path-based** — mount the extracted data and set the root; they
are auto-skipped (not failed) when absent, so the grid always runs over whatever is available.

## 0. Setup, configuration & reproducibility
All randomness flows through `set_seed`, and every knob lives in one `Config` object so a
run is fully described by `CFG`. This is the reproducibility backbone (Principle 4 / D7).

In [ ]:
# --- Colab installs (skip automatically off-Colab / if already present) -------------
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs], check=False)

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    _pip("datasets", "soundfile", "librosa", "ftfy", "regex", "openai-clip")
    # openai-clip exposes `import clip` and pulls ViT-B/32 weights on first use.
print("Colab:", IN_COLAB)

In [ ]:
import os, json, math, random, hashlib, copy, time, warnings
from dataclasses import dataclass, field, asdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed: int):
    "Make a run fully reproducible (fixes the 'no seeds' half of D7)."
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@dataclass
class Config:
    # reproducibility
    seed: int = 1234
    seeds: tuple = (0, 1, 2)          # >=3 seeds -> mean +/- 95% CI  (D7)
    smoke_test: bool = True           # tiny/fast end-to-end run; set False for real numbers
    work_dir: str = "/content" if (os.path.isdir("/content")) else "./work"
    # audio / spectrogram front-end
    sample_rate: int = 16000
    n_mels: int = 64
    n_fft: int = 1024
    hop_length: int = 256
    trim_top_db: float = 30.0         # silence trimming threshold (D6)
    max_seconds: float = 4.0          # fixed length AFTER trimming
    clip_norm: bool = True            # CLIP photographic mean/std (D5 ablation = Phase 3)
    # data
    cap_per_class: int = 1500         # balanced WaveFake subset: single-GPU tractable & reproducible
    split_fracs: dict = field(default_factory=lambda: {
        "train": 0.55, "dev": 0.10, "test": 0.25, "support": 0.10})
    # training
    batch_size: int = 16
    cnn_epochs: int = 8
    clip_epochs: int = 5
    lr_cnn: float = 1e-3
    lr_clip: float = 1e-4
    # few-shot
    few_shot_k: int = 50              # support examples PER CLASS
    few_shot_epochs: int = 20
    lr_finetune: float = 1e-3
    # metrics
    bootstrap_n: int = 1000

    @property
    def max_len(self) -> int:
        return int(self.sample_rate * self.max_seconds)

    def apply_smoke(self):
        "Shrink everything so the whole notebook runs end-to-end in a few minutes."
        if self.smoke_test:
            self.seeds = (0,)
            self.cap_per_class = 120
            self.cnn_epochs = 2
            self.clip_epochs = 1
            self.few_shot_k = 20
            self.few_shot_epochs = 8
            self.bootstrap_n = 200
        return self

CFG = Config().apply_smoke()
os.makedirs(CFG.work_dir, exist_ok=True)
set_seed(CFG.seed)
print("device:", DEVICE, "| smoke_test:", CFG.smoke_test)
print("seeds:", CFG.seeds, "| cap_per_class:", CFG.cap_per_class, "| max_len:", CFG.max_len)

## 1. Data acquisition

**WaveFake** (`ajaykarthick/wavefake-audio`) ships a single `train` split that mixes real
(`real_or_fake == "R"`) and fake clips. The original notebook (a) saved all ~60k rows and
(b) then *evaluated on them* — that is D1. Here we instead **stream** a small, **balanced,
deterministic** subset (first-N per class) to disk; §2 carves leak-free splits out of it.

**Fake-or-Real (FoR)** is optional and gated behind a Kaggle token. If you mount it, the
notebook adds a cross-corpus generalization number; otherwise it runs on WaveFake alone.

In [ ]:
# --- WaveFake: stream a balanced, deterministic subset to disk ----------------------
import soundfile as sf

WF_POOL = os.path.join(CFG.work_dir, "datasetWF", "all")   # <pool>/{real,fake}/*.wav

def _pool_counts(root):
    out = {}
    for c in ("real", "fake"):
        d = os.path.join(root, c)
        out[c] = len([f for f in os.listdir(d)]) if os.path.isdir(d) else 0
    return out

def build_wavefake_pool(cfg, pool_root=WF_POOL):
    "Stream first-N-per-class from the HF Hub into a fixed on-disk pool (deterministic)."
    for c in ("real", "fake"):
        os.makedirs(os.path.join(pool_root, c), exist_ok=True)
    cap = cfg.cap_per_class
    have = _pool_counts(pool_root)
    if have["real"] >= cap and have["fake"] >= cap:
        print("WaveFake pool already present:", have); return pool_root

    from datasets import load_dataset
    ds = load_dataset("ajaykarthick/wavefake-audio", split="train",
                      streaming=True, verification_mode="no_checks")
    n = {"real": have["real"], "fake": have["fake"]}
    for s in ds:
        if n["real"] >= cap and n["fake"] >= cap:
            break
        cat = "real" if s["real_or_fake"] == "R" else "fake"
        if n[cat] >= cap:
            continue
        arr = np.asarray(s["audio"]["array"], dtype=np.float32)
        sr = int(s["audio"]["sampling_rate"])
        fid = str(s.get("audio_id", f"{cat}_{n[cat]}"))
        sf.write(os.path.join(pool_root, cat, f"{fid}.wav"), arr, sr)
        n[cat] += 1
        if sum(n.values()) % 200 == 0:
            print("  collected", n)
    print("WaveFake pool ready:", _pool_counts(pool_root))
    return pool_root

# Heavy (network) — run once; cached on disk afterwards.
build_wavefake_pool(CFG)

In [ ]:
# --- (Optional) Fake-or-Real (FoR): point this at an extracted Kaggle copy ----------
# In Colab, the usual flow is:
#   from google.colab import files; files.upload()      # upload kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d mohammedabdeldayem/the-fake-or-real-dataset
#   !unzip -q the-fake-or-real-dataset.zip -d /content/for_raw
# FoR already provides training/validation/testing folders, each with real/ and fake/.
FOR_ROOT = os.path.join(CFG.work_dir, "for_raw")   # set to your extracted path, or leave missing
HAS_FOR = os.path.isdir(FOR_ROOT)
print("FoR available:", HAS_FOR, "->", FOR_ROOT)

## 2. Leak-free deterministic splits  — fixes **D1 / D2**

We build **disjoint** `train / dev / test / support` manifests from each pool, **stratified
by class** and driven by a fixed seed so the split is identical on every machine. Manifests
record a SHA-256 per file plus a manifest-level hash (provenance + reproducibility), and we
**assert the four index sets are pairwise disjoint**. Downstream, evaluation reads **`test`
only** — never `train`.

Label convention (standardised; the original notebook was inconsistent):
**`1 = real / bonafide`, `0 = fake / spoof`**, and every decision *score is higher for real*.

In [ ]:
LABELS = {"fake": 0, "real": 1}                 # bonafide(real)=1, spoof(fake)=0  (anti-spoofing convention)
SPLITS = ("train", "dev", "test", "support")

def _sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def _list_pool(pool_root):
    "Deterministic, sorted (path,label) listing of a {real,fake} pool."
    items = []
    for cat, lab in LABELS.items():
        d = os.path.join(pool_root, cat)
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            items.append((os.path.join(d, fn), lab))
    return sorted(items)

def build_splits(pool_root, cfg, name, with_hashes=True):
    """Stratified, seeded, DISJOINT split -> dict[split] -> list[{path,label,sha256}].
    Fixes D1/D2: produces a real held-out `test`, fully disjoint from `train`/`dev`/`support`."""
    items = _list_pool(pool_root)
    assert items, f"empty pool: {pool_root}"
    fr = cfg.split_fracs
    rng = random.Random(cfg.seed)
    per_split = {s: [] for s in SPLITS}
    for lab in sorted(set(LABELS.values())):                  # stratify per class
        group = [p for (p, l) in items if l == lab]
        rng.shuffle(group)                                    # seeded
        n = len(group)
        n_tr = int(fr["train"] * n)
        n_dv = int(fr["dev"] * n)
        n_te = int(fr["test"] * n)
        bounds = {
            "train":   group[:n_tr],
            "dev":     group[n_tr:n_tr + n_dv],
            "test":    group[n_tr + n_dv:n_tr + n_dv + n_te],
            "support": group[n_tr + n_dv + n_te:],            # remainder
        }
        for s in SPLITS:
            per_split[s] += [{"path": p, "label": lab} for p in bounds[s]]

    # --- disjointness guarantee (the heart of the D1/D2 fix) ---
    sets = {s: set(e["path"] for e in per_split[s]) for s in SPLITS}
    for a in SPLITS:
        for b in SPLITS:
            if a < b:
                inter = sets[a] & sets[b]
                assert not inter, f"LEAK: {name}/{a} overlaps {name}/{b} ({len(inter)})"

    if with_hashes:
        for s in SPLITS:
            for e in per_split[s]:
                e["sha256"] = _sha256_file(e["path"])
    manifest = {"dataset": name, "seed": cfg.seed, "label_map": LABELS,
                "counts": {s: len(per_split[s]) for s in SPLITS}, "splits": per_split}
    manifest["manifest_sha256"] = hashlib.sha256(
        json.dumps(manifest["splits"], sort_keys=True).encode()).hexdigest()[:16]
    out = os.path.join(cfg.work_dir, f"splits_{name}.json")
    with open(out, "w") as f:
        json.dump(manifest, f)
    print(f"[{name}] counts={manifest['counts']}  hash={manifest['manifest_sha256']}  -> {out}")
    return manifest

WF_SPLITS = build_splits(WF_POOL, CFG, "wavefake")

In [ ]:
# --- (Optional) FoR manifests from its NATIVE train/val/test folders ----------------
def build_for_manifest(for_root, cfg):
    "FoR ships its own splits; map them to our schema (training->train, validation->dev, testing->test)."
    name_map = {"training": "train", "validation": "dev", "testing": "test"}
    per_split = {s: [] for s in SPLITS}
    for native, ours in name_map.items():
        for cat, lab in LABELS.items():
            d = os.path.join(for_root, native, cat)
            if not os.path.isdir(d):
                continue
            for fn in sorted(os.listdir(d)):
                if fn.lower().endswith((".wav", ".flac", ".mp3")):
                    per_split[ours].append({"path": os.path.join(d, fn), "label": lab})
    sets = {s: set(e["path"] for e in per_split[s]) for s in SPLITS}
    for a in SPLITS:
        for b in SPLITS:
            if a < b:
                assert not (sets[a] & sets[b]), f"FoR leak {a}/{b}"
    man = {"dataset": "for", "counts": {s: len(per_split[s]) for s in SPLITS}, "splits": per_split}
    print("[for] counts=", man["counts"])
    return man

FOR_SPLITS = build_for_manifest(FOR_ROOT, CFG) if HAS_FOR else None

## 3. Preprocessing with silence-trimming — mitigates **D6**

The original fixed-length pad/trim to 25 000 samples injected **zero-padding silence** and a
**clip-duration** cue; if real and fake corpora differ in length, the model can separate them
without ever looking at deepfake artifacts. We therefore **trim leading/trailing silence
first**, then pad/trim to a fixed length. We also record per-clip metadata
(`orig_len`, `silence_ratio`, `orig_sr`) so §9 can *prove* the detector isn't keying on them.

In [ ]:
import librosa
import torchaudio

_MEL = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG.sample_rate, n_mels=CFG.n_mels, n_fft=CFG.n_fft, hop_length=CFG.hop_length)
_CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
_CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

def load_waveform(path, cfg):
    "Load -> mono -> resample to cfg.sample_rate. Returns (wave float32 [L], meta)."
    y, sr = librosa.load(path, sr=None, mono=True)             # native sr first (for meta)
    meta = {"orig_sr": int(sr), "orig_len": int(len(y))}
    if sr != cfg.sample_rate:
        y = librosa.resample(y, orig_sr=sr, target_sr=cfg.sample_rate)
    return y.astype(np.float32), meta

def trim_silence(y, cfg):
    "Strip leading/trailing silence (D6). Returns (trimmed, silence_ratio_removed)."
    if len(y) == 0:
        return y, 0.0
    yt, _ = librosa.effects.trim(y, top_db=cfg.trim_top_db)
    if len(yt) == 0:                                            # all-silence guard
        yt = y
    silence_ratio = 1.0 - (len(yt) / max(1, len(y)))
    return yt.astype(np.float32), float(silence_ratio)

def fix_length(y, cfg):
    L = cfg.max_len
    if len(y) < L:
        y = np.pad(y, (0, L - len(y)))
    else:
        y = y[:L]
    return y.astype(np.float32)

def waveform_for_cnn(path, cfg):
    "Raw 1-channel waveform for the CNN-LSTM, with silence trimmed. -> ([1,L] tensor, meta)."
    y, meta = load_waveform(path, cfg)
    y, sr_ratio = trim_silence(y, cfg)
    meta["silence_ratio"] = sr_ratio
    meta["trimmed_len"] = int(len(y))
    y = fix_length(y, cfg)
    return torch.from_numpy(y).unsqueeze(0), meta

def mel_for_clip(path, cfg):
    "3x224x224 mel 'image' for CLIP, with silence trimmed. -> ([3,224,224] tensor, meta)."
    wav, meta = waveform_for_cnn(path, cfg)                     # reuse trim+fix
    mel = _MEL(wav)                                             # [1, n_mels, T]
    mel = torch.log1p(mel)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    mel = mel.repeat(3, 1, 1)                                   # 3-channel
    mel = F.interpolate(mel.unsqueeze(0), size=(224, 224), mode="bilinear",
                        align_corners=False).squeeze(0)
    if cfg.clip_norm:                                           # CLIP photographic norm (D5: studied in Phase 3)
        for c in range(3):
            mel[c] = (mel[c] - _CLIP_MEAN[c]) / _CLIP_STD[c]
    return mel, meta

In [ ]:
class ManifestDataset(Dataset):
    """Reads a manifest split. mode='cnn' -> raw waveform; mode='clip' -> mel image.
    Returns (tensor, label, meta) so eval can also audit shortcuts (D6)."""
    def __init__(self, entries, cfg, mode="cnn"):
        self.entries = entries
        self.cfg = cfg
        self.mode = mode

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, i):
        e = self.entries[i]
        fn = waveform_for_cnn if self.mode == "cnn" else mel_for_clip
        try:
            x, meta = fn(e["path"], self.cfg)
        except Exception as ex:                                 # never crash a long run on one bad file
            print("skip", e["path"], ex)
            shape = (1, self.cfg.max_len) if self.mode == "cnn" else (3, 224, 224)
            x, meta = torch.zeros(*shape), {"orig_sr": 0, "orig_len": 0, "silence_ratio": 0.0, "trimmed_len": 0}
        meta["label"] = e["label"]
        return x, e["label"], meta

def make_loader(manifest, split, cfg, mode="cnn", shuffle=False, batch_size=None):
    ds = ManifestDataset(manifest["splits"][split], cfg, mode=mode)
    return DataLoader(ds, batch_size=batch_size or cfg.batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=(DEVICE == "cuda"), drop_last=False)

# sanity: one batch
_dl = make_loader(WF_SPLITS, "test", CFG, mode="cnn")
_xb, _yb, _mb = next(iter(_dl))
print("cnn batch:", _xb.shape, "labels:", _yb[:8].tolist())

## 4. Models
Two detectors, unchanged in spirit from the original so the corrected numbers are comparable:
a **CNN-LSTM** on raw waveform (the baseline), and the **CLIP ViT-B/32 + MultiLevelAdapter +
FC head** on mel "images" (the studied method). CLIP's backbone is frozen — only the adapter
and head train, which keeps the whole thing single-GPU friendly.

In [ ]:
class CNN_LSTM(nn.Module):
    "Raw-waveform baseline (1D conv -> BiLSTM -> FC). Output logits over [fake, real]."
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2)
        self.pool = nn.MaxPool1d(2, 2)
        self.lstm = nn.LSTM(32, 64, num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        x = x.float()
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.permute(0, 2, 1)               # [B, T, C]
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])

In [ ]:
# CLIP is imported lazily so §0-§3 work even before the (heavier) CLIP weights are pulled.
import clip as openai_clip

class MultiLevelAdapter(nn.Module):
    "Residual feature adapter on top of frozen CLIP image features (input/output dim 512)."
    def __init__(self, input_dim=512, output_dim=256, gamma=0.1):
        super().__init__()
        self.adapter1 = nn.Sequential(nn.Linear(input_dim, output_dim), nn.ReLU(),
                                      nn.Linear(output_dim, output_dim))
        self.adapter2 = nn.Sequential(nn.Linear(output_dim, output_dim), nn.ReLU(),
                                      nn.Linear(output_dim, output_dim))
        self.projection = nn.Linear(output_dim, input_dim)
        self.gamma = gamma

    def forward(self, x):
        x2 = self.adapter2(self.adapter1(x))
        return self.gamma * self.projection(x2) + (1 - self.gamma) * x

class CLIPFineTuner(nn.Module):
    "Frozen CLIP image encoder -> MultiLevelAdapter -> FC head. Output logits over [fake, real]."
    def __init__(self, clip_model):
        super().__init__()
        for p in clip_model.parameters():
            p.requires_grad = False
        self.clip_model = clip_model.eval()
        self.adapter = MultiLevelAdapter(512, 256)
        self.fc = nn.Linear(512, 2)

    def forward(self, image):
        with torch.no_grad():
            feats = self.clip_model.encode_image(image).float()
        return self.fc(self.adapter(feats))

def load_clip_backbone():
    model, _ = openai_clip.load("ViT-B/32", device=DEVICE)
    return model

print("CLIP loader ready (weights download on first build_clip()).")

## 5. Metrics — fixes **D7**

The field standard for anti-spoofing is **EER** (primary) with **bootstrap confidence
intervals**, reported over **multiple seeds** — not a single AUC. This section provides:

* `compute_eer`, `compute_auc`, `balanced_acc`, and calibration (`brier`, `ece`);
* `bootstrap_ci` — percentile 95% CI for any metric on one run;
* `aggregate_seeds` — mean ± 95% CI **across ≥3 seeds**;
* `compute_min_tdcf` — the official ASVspoof2019 min t-DCF, included and unit-checkable but
  **inactive in Phase 1** because WaveFake/FoR ship no ASV scores; it activates in Phase 2
  with ASVspoof (a-DCF likewise). This is the honest reading of the plan: t-DCF/a-DCF are
  *"for ASVspoof"*, EER/AUC are what WaveFake/FoR support.

Score convention everywhere: **higher score ⇒ more "real" (bonafide)**, `label 1 = real`.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, balanced_accuracy_score, brier_score_loss

def compute_auc(labels, scores):
    labels = np.asarray(labels);
    if len(np.unique(labels)) < 2:
        return float("nan")
    return float(roc_auc_score(labels, scores))

def compute_eer(labels, scores):
    "Equal Error Rate (primary anti-spoofing metric). pos_label=1=real."
    labels = np.asarray(labels)
    if len(np.unique(labels)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2.0)

def balanced_acc(labels, scores, thr=0.5):
    "Balanced accuracy at a probability threshold (expects calibrated P(real) in [0,1])."
    labels = np.asarray(labels)
    preds = (np.asarray(scores) >= thr).astype(int)
    return float(balanced_accuracy_score(labels, preds))

def brier(labels, probs):
    "Brier score on P(real); lower=better calibrated."
    labels = np.asarray(labels)
    if len(np.unique(labels)) < 2:
        return float("nan")
    return float(brier_score_loss(labels, np.clip(probs, 0, 1)))

def ece(labels, probs, n_bins=10):
    "Expected Calibration Error on P(real)."
    labels = np.asarray(labels); probs = np.clip(np.asarray(probs), 0, 1)
    bins = np.linspace(0, 1, n_bins + 1)
    e, N = 0.0, len(labels)
    for b in range(n_bins):
        m = (probs > bins[b]) & (probs <= bins[b + 1]) if b > 0 else (probs >= bins[0]) & (probs <= bins[1])
        if m.sum() == 0:
            continue
        e += (m.sum() / N) * abs(labels[m].mean() - probs[m].mean())
    return float(e)

In [ ]:
def bootstrap_ci(labels, scores, metric_fn, n=1000, seed=0, alpha=0.05):
    "Percentile bootstrap 95% CI for any metric_fn(labels, scores) on a single run."
    labels = np.asarray(labels); scores = np.asarray(scores)
    rng = np.random.default_rng(seed)
    N = len(labels); vals = []
    for _ in range(n):
        idx = rng.integers(0, N, N)
        if len(np.unique(labels[idx])) < 2:
            continue
        vals.append(metric_fn(labels[idx], scores[idx]))
    vals = np.asarray(vals, dtype=float)
    if vals.size == 0:
        return (float("nan"), float("nan"), float("nan"))
    return (float(np.mean(vals)),
            float(np.percentile(vals, 100 * alpha / 2)),
            float(np.percentile(vals, 100 * (1 - alpha / 2))))

def evaluate_scores(labels, scores, probs, cfg, tag=""):
    "Full metric bundle for one run, each headline metric with a bootstrap 95% CI."
    out = {"tag": tag, "n": int(len(labels)),
           "auc": compute_auc(labels, scores),
           "eer": compute_eer(labels, scores),
           "bal_acc": balanced_acc(labels, probs),
           "brier": brier(labels, probs),
           "ece": ece(labels, probs)}
    out["auc_ci"] = bootstrap_ci(labels, scores, compute_auc, cfg.bootstrap_n, cfg.seed)[1:]
    out["eer_ci"] = bootstrap_ci(labels, scores, compute_eer, cfg.bootstrap_n, cfg.seed)[1:]
    return out

def aggregate_seeds(runs, keys=("auc", "eer", "bal_acc")):
    "Mean +/- 95% CI ACROSS seeds (t-interval). Fulfils the '>=3 seeds, mean +/- 95% CI' criterion."
    from math import sqrt
    agg = {}
    for k in keys:
        xs = np.asarray([r[k] for r in runs if not math.isnan(r.get(k, float('nan')))], dtype=float)
        if xs.size == 0:
            agg[k] = (float("nan"), float("nan")); continue
        m = float(xs.mean())
        if xs.size == 1:
            agg[k] = (m, 0.0)
        else:
            # 95% t-interval half-width
            tcrit = {2: 12.71, 3: 4.303, 4: 3.182, 5: 2.776}.get(xs.size, 1.96)
            agg[k] = (m, float(tcrit * xs.std(ddof=1) / sqrt(xs.size)))
    agg["n_seeds"] = len(runs)
    return agg

In [ ]:
# ---- Official ASVspoof2019 DET / EER / min t-DCF (faithful port; t-DCF is Phase-2) ----
def compute_det_curve(target_scores, nontarget_scores):
    "Canonical anti-spoofing DET curve (ASVspoof eval_metrics.py)."
    target_scores = np.asarray(target_scores); nontarget_scores = np.asarray(nontarget_scores)
    n = target_scores.size + nontarget_scores.size
    all_scores = np.concatenate((target_scores, nontarget_scores))
    labels = np.concatenate((np.ones(target_scores.size), np.zeros(nontarget_scores.size)))
    idx = np.argsort(all_scores, kind="mergesort"); labels = labels[idx]
    tar_sums = np.cumsum(labels)
    nontar_sums = nontarget_scores.size - (np.arange(1, n + 1) - tar_sums)
    frr = np.concatenate((np.atleast_1d(0), tar_sums / target_scores.size))
    far = np.concatenate((np.atleast_1d(1), nontar_sums / nontarget_scores.size))
    thr = np.concatenate((np.atleast_1d(all_scores[idx[0]] - 1e-3), all_scores[idx]))
    return frr, far, thr

def compute_eer_official(target_scores, nontarget_scores):
    "EER via DET intersection (matches the ASVspoof reference)."
    frr, far, thr = compute_det_curve(target_scores, nontarget_scores)
    i = int(np.nanargmin(np.abs(frr - far)))
    return float(np.mean((frr[i], far[i]))), float(thr[i])

def compute_min_tdcf(bonafide_cm, spoof_cm, Pfa_asv, Pmiss_asv, Pmiss_spoof_asv,
                     cost_model=None):
    """Faithful ASVspoof2019 min-normalised t-DCF (Kinnunen et al., 2018).
    PHASE-1 INACTIVE: needs an ASV system's error rates (Pfa_asv, Pmiss_asv, Pmiss_spoof_asv),
    which WaveFake/FoR don't provide. Activated in Phase 2 with ASVspoof19/21 LA; a-DCF
    (ASVspoof 5) is added alongside via the official tooling."""
    if cost_model is None:
        cost_model = {"Pspoof": 0.05, "Ptar": 0.95 * 0.99, "Pnon": 0.95 * 0.01,
                      "Cmiss": 1.0, "Cfa": 10.0, "Cfa_spoof": 10.0}
    frr, far, thr = compute_det_curve(np.asarray(bonafide_cm), np.asarray(spoof_cm))
    Pmiss_cm, Pfa_cm = frr, far
    C0 = cost_model["Ptar"] * cost_model["Cmiss"] * Pmiss_asv \
        + cost_model["Pnon"] * cost_model["Cfa"] * Pfa_asv
    C1 = cost_model["Ptar"] * cost_model["Cmiss"] - C0
    C2 = cost_model["Pspoof"] * cost_model["Cfa_spoof"] * (1 - Pmiss_spoof_asv)
    tdcf = C0 + C1 * Pmiss_cm + C2 * Pfa_cm
    denom = min(C0 + C1, C0 + C2)
    return float(np.min(tdcf) / denom) if denom > 0 else float("nan")

# self-test: EER ~0 for separable, ~0.5 for random; both EER implementations agree
_lab = np.array([1, 1, 1, 0, 0, 0]); _sep = np.array([0.9, 0.8, 0.7, 0.2, 0.1, 0.05])
assert compute_eer(_lab, _sep) < 1e-6, "EER self-test failed (separable)"
assert abs(compute_auc(_lab, _sep) - 1.0) < 1e-6, "AUC self-test failed (separable)"
_eo, _ = compute_eer_official(_sep[_lab == 1], _sep[_lab == 0])
assert _eo < 1e-6, "official EER self-test failed"
print("metrics self-test OK | EER=%.3f AUC=%.3f official-EER=%.3f" % (
    compute_eer(_lab, _sep), compute_auc(_lab, _sep), _eo))

## 6. Training (seeded)
Trainers are plain and deterministic. Each takes a `seed` so the multi-seed runner (§10) can
produce mean ± CI. Models output logits over `[fake, real]`; `P(real) = softmax(...)[:, 1]`.

In [ ]:
def train_cnn(manifest, cfg, seed):
    set_seed(seed)
    model = CNN_LSTM().to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr_cnn)
    crit = nn.CrossEntropyLoss()
    loader = make_loader(manifest, "train", cfg, mode="cnn", shuffle=True)
    model.train()
    for ep in range(cfg.cnn_epochs):
        tot, nb = 0.0, 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.long().to(DEVICE)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); tot += loss.item(); nb += 1
        print(f"  [cnn seed{seed}] epoch {ep+1}/{cfg.cnn_epochs} loss={tot/max(1,nb):.4f}")
    return model

def build_clip():
    return CLIPFineTuner(load_clip_backbone()).to(DEVICE)

def train_clip(manifest, cfg, seed):
    set_seed(seed)
    model = build_clip()
    params = [p for p in model.parameters() if p.requires_grad]   # adapter + fc only
    opt = optim.Adam(params, lr=cfg.lr_clip)
    crit = nn.CrossEntropyLoss()
    loader = make_loader(manifest, "train", cfg, mode="clip", shuffle=True)
    model.train()
    for ep in range(cfg.clip_epochs):
        tot, nb = 0.0, 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.long().to(DEVICE)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        print(f"  [clip seed{seed}] epoch {ep+1}/{cfg.clip_epochs} loss={tot/max(1,nb):.4f}")
    return model

## 7. Leak-free evaluation harness
Every collector returns `P(real)` in `[0, 1]`, used as both the ranking score (EER/AUC) and
the calibrated probability (Brier/ECE). **All collectors are pointed at the `test` split only.**

In [ ]:
@torch.no_grad()
def collect_logit_probs(model, loader):
    "For CNN / CLIP-head models: P(real)=softmax(logits)[:,1]."
    model.eval(); labels, probs = [], []
    for x, y, _ in loader:
        x = x.to(DEVICE)
        p = torch.softmax(model(x), dim=1)[:, 1]
        probs.extend(p.cpu().numpy().tolist()); labels.extend(y.numpy().tolist())
    return np.array(labels), np.array(probs)

@torch.no_grad()
def collect_clip_zeroshot(clip_backbone, loader, device=DEVICE):
    "Untrained CLIP text-image cosine: P(real) from softmax over [sim_fake, sim_real]. Expect ~chance (D5)."
    clip_backbone.eval()
    real_prompts = ["a real audio clip", "an authentic voice recording", "a genuine human speech sample"]
    fake_prompts = ["a fake audio clip", "a synthetic deepfake voice", "an AI-generated speech sample"]
    rt = openai_clip.tokenize(real_prompts).to(device)
    ft = openai_clip.tokenize(fake_prompts).to(device)
    te_real = clip_backbone.encode_text(rt).float().mean(0, keepdim=True)
    te_fake = clip_backbone.encode_text(ft).float().mean(0, keepdim=True)
    te_real = te_real / te_real.norm(dim=-1, keepdim=True)
    te_fake = te_fake / te_fake.norm(dim=-1, keepdim=True)
    text = torch.cat([te_fake, te_real], 0)                       # row0=fake, row1=real
    labels, probs = [], []
    for x, y, _ in loader:
        x = x.to(device)
        ie = clip_backbone.encode_image(x).float()
        ie = ie / ie.norm(dim=-1, keepdim=True)
        sims = 100.0 * ie @ text.T                                # [B,2]
        p = torch.softmax(sims, dim=1)[:, 1]                      # P(real)
        probs.extend(p.cpu().numpy().tolist()); labels.extend(y.numpy().tolist())
    return np.array(labels), np.array(probs)

def evaluate_model(model_or_backbone, manifest, cfg, kind, tag):
    "kind in {'cnn','clip_head','clip_zeroshot'}. Reads TEST split only (D1/D2)."
    mode = "cnn" if kind == "cnn" else "clip"
    loader = make_loader(manifest, "test", cfg, mode=mode)
    if kind == "clip_zeroshot":
        labels, probs = collect_clip_zeroshot(model_or_backbone, loader)
    else:
        labels, probs = collect_logit_probs(model_or_backbone, loader)
    return evaluate_scores(labels, probs, probs, cfg, tag=tag)

## 8. Few-shot — fixes **D3** (leakage) and **D4** (frozen loss)

* **D3:** `disjoint_support_query` selects *k examples per class* from the **`support`** split
  (which §2 already built disjoint from `test`) and uses **`test`** as the query, then
  **asserts `support ∩ query == ∅`**. The original drew both with `random.sample` from one
  pool, so they overlapped — that is what inflated the 0.85.
* **D4:** `finetune_cnn_few_shot` freezes the backbone, builds the optimizer over **only the
  unfrozen head params of *this* model**, and **asserts the loss strictly decreases**. The
  original passed a stale `optimizer` bound to a different model object, so nothing learned
  (loss stuck at 1.7984).

In [ ]:
def disjoint_support_query(support_entries, query_entries, k_per_class, seed):
    "D3 fix: k-per-class support from the support split; test split as query; assert disjoint."
    rng = random.Random(seed)
    by = {0: [], 1: []}
    for e in support_entries:
        by[e["label"]].append(e)
    support = []
    for lab in (0, 1):
        g = sorted(by[lab], key=lambda e: e["path"]); rng.shuffle(g)
        support += g[:k_per_class]
    query = list(query_entries)
    s_paths = set(e["path"] for e in support); q_paths = set(e["path"] for e in query)
    assert s_paths.isdisjoint(q_paths), "D3 VIOLATION: support and query overlap!"
    return support, query

def finetune_cnn_few_shot(base_model, support_entries, cfg, seed, freeze_backbone=True):
    "D4 fix: optimizer over unfrozen params of the (copied) loaded model; assert loss decreases."
    set_seed(seed)
    model = copy.deepcopy(base_model).to(DEVICE)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
        for p in model.fc.parameters():
            p.requires_grad = True
    params = [p for p in model.parameters() if p.requires_grad]
    assert params, "no trainable params after freezing"
    opt = optim.Adam(params, lr=cfg.lr_finetune)
    crit = nn.CrossEntropyLoss()
    loader = DataLoader(ManifestDataset(support_entries, cfg, mode="cnn"),
                        batch_size=cfg.batch_size, shuffle=True)
    model.train(); losses = []
    for ep in range(cfg.few_shot_epochs):
        tot, nb = 0.0, 0
        for x, y, _ in loader:
            x, y = x.to(DEVICE), y.long().to(DEVICE)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        losses.append(tot / max(1, nb))
    assert losses[-1] < losses[0] - 1e-4, f"D4 VIOLATION: few-shot loss did not decrease: {losses}"
    return model, losses

@torch.no_grad()
def clip_prototype_few_shot(clip_head_model, support_entries, query_entries, cfg):
    "Prototype few-shot for CLIP: class prototypes from support image features, cosine score on query."
    clip = clip_head_model.clip_model.eval()
    def emb(entries):
        loader = DataLoader(ManifestDataset(entries, cfg, mode="clip"), batch_size=cfg.batch_size)
        E, Y = [], []
        for x, y, _ in loader:
            e = clip.encode_image(x.to(DEVICE)).float()
            e = e / e.norm(dim=-1, keepdim=True)
            E.append(e.cpu()); Y.extend(y.numpy().tolist())
        return torch.cat(E, 0), np.array(Y)
    se, sy = emb(support_entries)
    proto_fake = se[sy == 0].mean(0, keepdim=True); proto_fake /= proto_fake.norm(dim=-1, keepdim=True)
    proto_real = se[sy == 1].mean(0, keepdim=True); proto_real /= proto_real.norm(dim=-1, keepdim=True)
    qe, qy = emb(query_entries)
    sim = qe @ torch.cat([proto_fake, proto_real], 0).T          # [N,2]
    probs = torch.softmax(100.0 * sim, dim=1)[:, 1].numpy()      # P(real)
    return qy, probs

## 9. Shortcut probe baselines — audits **D6**

If a trivial classifier using **only** clip duration, silence ratio, or sample rate can beat
chance, the "deepfake detector" may be exploiting that shortcut instead of artifacts. After
silence-trimming (§3), these probes **should score ≈ 0.5 AUC**. Each probe trains a logistic
regression on the `train` split's metadata and reports AUC on `test`; `|AUC − 0.5| > 0.05`
raises a shortcut flag.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def _meta_features(entries, cfg, cap=None):
    "Per-clip [orig_len, silence_ratio, orig_sr] + labels. cap limits cost on big splits."
    if cap is not None and len(entries) > cap:
        entries = entries[:cap]
    X, y = [], []
    for e in entries:
        wav, meta = load_waveform(e["path"], cfg)
        _, sil = trim_silence(wav, cfg)
        X.append([meta["orig_len"], sil, meta["orig_sr"]])
        y.append(e["label"])
    return np.asarray(X, dtype=float), np.asarray(y)

PROBE_COLS = {"duration_only": [0], "silence_only": [1], "samplerate_only": [2], "all_meta": [0, 1, 2]}

def run_shortcut_probes(manifest, cfg, cap=300):
    Xtr, ytr = _meta_features(manifest["splits"]["train"], cfg, cap)
    Xte, yte = _meta_features(manifest["splits"]["test"], cfg, cap)
    results = {}
    for name, cols in PROBE_COLS.items():
        if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2:
            results[name] = float("nan"); continue
        sc = StandardScaler().fit(Xtr[:, cols])
        clf = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr[:, cols]), ytr)
        p = clf.predict_proba(sc.transform(Xte[:, cols]))[:, 1]
        auc = compute_auc(yte, p)
        flag = "SHORTCUT?" if abs(auc - 0.5) > 0.05 else "ok(~chance)"
        results[name] = auc
        print(f"  probe {name:16s} AUC={auc:.3f}  {flag}")
    return results

## 12. Common dataset schema & provenance

Every adapter returns a manifest in the **exact Phase-1 format** (`splits -> train/dev/test/support
-> [{path,label,...}]`) so the Phase-1 harness works unchanged. Two additions:
* a `provenance` block (source/license/version/date) attached to each manifest, and
* **group-disjoint splitting** (`build_splits_grouped`) so a *speaker* (In-the-Wild) or *generator*
  (MLAAD/ASVspoof) never spans train and test — the leakage that matters for a generalization claim.

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class Provenance:
    name: str
    source_url: str = "VERIFY"
    license: str = "VERIFY"
    version: str = "VERIFY"
    access_date: str = "VERIFY"
    citation: str = "VERIFY"
    notes: str = ""
    verified: bool = False            # flip to True only after checking the primary source

def build_splits_grouped(items, cfg, name, group_key="group", with_hashes=False):
    """Group-disjoint stratified split: whole groups (speaker/generator) are assigned to a single
    split, so train/dev/test/support share no group. Preserves every entry field (incl. 'meta')."""
    fr = cfg.split_fracs
    rng = random.Random(cfg.seed)
    groups = {}
    for e in items:
        groups.setdefault(e[group_key], []).append(e)
    gids = sorted(groups); rng.shuffle(gids)
    total = max(1, len(items))
    targets = {s: fr[s] for s in SPLITS}
    per = {s: [] for s in SPLITS}
    counts = {s: 0 for s in SPLITS}
    for g in gids:                                   # greedily fill the split with the largest deficit
        best = max(SPLITS, key=lambda s: targets[s] * total - counts[s])
        per[best] += groups[g]; counts[best] += len(groups[g])
    gset = {s: set(e[group_key] for e in per[s]) for s in SPLITS}
    for a in SPLITS:
        for b in SPLITS:
            if a < b:
                assert gset[a].isdisjoint(gset[b]), f"GROUP LEAK {name}: {a} & {b} share a group"
    if with_hashes:
        for s in SPLITS:
            for e in per[s]:
                if "sha256" not in e and os.path.isfile(e["path"]):
                    e["sha256"] = _sha256_file(e["path"])
    for s in ("train", "test"):
        labs = set(e["label"] for e in per[s])
        if labs != {0, 1}:
            print(f"  WARN [{name}] split '{s}' has labels {labs} (expected both classes)")
    man = {"dataset": name, "seed": cfg.seed, "label_map": LABELS,
           "counts": {s: len(per[s]) for s in SPLITS}, "splits": per,
           "grouped_by": group_key}
    man["manifest_sha256"] = hashlib.sha256(
        json.dumps({s: sorted(e["path"] for e in per[s]) for s in SPLITS}, sort_keys=True).encode()
    ).hexdigest()[:16]
    print(f"[{name}] grouped counts={man['counts']} groups={sum(len(v) for v in gset.values())} hash={man['manifest_sha256']}")
    return man

## 13. Dataset loaders (adapters)

Each adapter exposes `available()` and `build_manifest(cfg)`. HF adapters stream a balanced,
deterministic, **capped** subset to disk (same approach as Phase 1's WaveFake loader); path-based
adapters read mounted data using official protocol files.

In [ ]:
import soundfile as sf

def stream_hf_pool(hf_id, name, label_fn, cfg, split="train", audio_key="audio", id_key="audio_id"):
    "Stream first-N-per-class from an HF dataset into a fixed {real,fake} pool (deterministic)."
    pool_root = os.path.join(cfg.work_dir, f"pool_{name}")
    for c in ("real", "fake"):
        os.makedirs(os.path.join(pool_root, c), exist_ok=True)
    cap = cfg.cap_per_class
    have = {c: len(os.listdir(os.path.join(pool_root, c))) for c in ("real", "fake")}
    if have["real"] >= cap and have["fake"] >= cap:
        print(f"[{name}] pool cached: {have}"); return pool_root
    from datasets import load_dataset
    ds = load_dataset(hf_id, split=split, streaming=True, verification_mode="no_checks")
    n = dict(have)
    for s in ds:
        if n["real"] >= cap and n["fake"] >= cap:
            break
        cat = label_fn(s)
        if cat is None or n[cat] >= cap:
            continue
        arr = np.asarray(s[audio_key]["array"], dtype=np.float32)
        sr = int(s[audio_key]["sampling_rate"])
        fid = str(s.get(id_key, f"{cat}_{n[cat]}"))
        sf.write(os.path.join(pool_root, cat, f"{fid}.wav"), arr, sr); n[cat] += 1
        if sum(n.values()) % 200 == 0:
            print(f"  [{name}] collected {n}")
    print(f"[{name}] pool ready: {n}")
    return pool_root

def wavefake_label(s):
    return "real" if s.get("real_or_fake") == "R" else "fake"

def tolerant_label(s):
    "Schema-tolerant real/fake detection for same-family HF sets (e.g. codecfake-audio). VERIFY."
    for k in ("real_or_fake", "label", "is_fake", "type", "class", "target"):
        if k in s:
            v = str(s[k]).strip().lower()
            if v in ("r", "real", "bonafide", "bona-fide", "genuine", "0"):
                return "real"
            if v in ("f", "fake", "spoof", "1"):
                return "fake"
    return None

class HFStreamAdapter:
    def __init__(self, name, hf_id, label_fn, provenance, enabled=True):
        self.name, self.hf_id, self.label_fn = name, hf_id, label_fn
        self.provenance, self.enabled = provenance, enabled
    def available(self):
        return self.enabled            # assumes HF reachable (true on Colab); toggle via P2_ENABLE
    def build_manifest(self, cfg):
        pool = stream_hf_pool(self.hf_id, self.name, self.label_fn, cfg)
        man = build_splits(pool, cfg, self.name)
        man["provenance"] = asdict(self.provenance)
        return man

In [ ]:
import csv

class InTheWildAdapter:
    """In-the-Wild: speaker-disjoint splits. Prefers a locally extracted copy with meta.csv
    (columns: file,speaker,label where label in {bona-fide,spoof}); falls back to HF streaming
    (no speaker -> plain split) if only the HF id is reachable."""
    def __init__(self, provenance, local_root=None, hf_id="mueller91/In-The-Wild", enabled=True):
        self.provenance, self.local_root, self.hf_id, self.enabled = provenance, local_root, hf_id, enabled
    def available(self):
        return self.enabled or (self.local_root and os.path.isdir(self.local_root))
    def _items_from_meta(self, cfg):
        meta = os.path.join(self.local_root, "meta.csv")
        items = []
        with open(meta) as f:
            for row in csv.DictReader(f):
                lab_raw = (row.get("label") or row.get("class") or "").strip().lower()
                lab = 1 if lab_raw in ("bona-fide", "bonafide", "real") else 0
                fn = row.get("file") or row.get("filename") or row.get("path")
                spk = row.get("speaker") or row.get("speaker_id") or fn
                p = fn if os.path.isabs(fn) else os.path.join(self.local_root, fn)
                items.append({"path": p, "label": lab, "group": str(spk),
                              "meta": {"speaker": str(spk), "generator": None}})
        return items
    def build_manifest(self, cfg):
        if self.local_root and os.path.isfile(os.path.join(self.local_root, "meta.csv")):
            man = build_splits_grouped(self._items_from_meta(cfg), cfg, "inthewild", group_key="group")
        else:  # HF fallback: stream a capped pool, plain (non-speaker) split + a warning
            print("[inthewild] no local meta.csv -> HF streaming, plain split (speaker-disjointness unavailable)")
            pool = stream_hf_pool(self.hf_id, "inthewild", tolerant_label, cfg)
            man = build_splits(pool, cfg, "inthewild")
        man["provenance"] = asdict(self.provenance)
        return man

class FoRAdapter:
    "Fake-or-Real: native train/val/test folders (reuses the Phase-1 builder)."
    def __init__(self, provenance, root):
        self.provenance, self.root = provenance, root
    def available(self):
        return os.path.isdir(self.root)
    def build_manifest(self, cfg):
        man = build_for_manifest(self.root, cfg)
        man["provenance"] = asdict(self.provenance)
        return man

In [ ]:
class MLAADAdapter:
    """MLAAD (spoof-only): fakes laid out as <root>/fake/<language>/<tts_model>/*.wav, so each file's
    generator + language come from its path -> ready for LOGO. Real speech is supplied separately
    (M-AILABS or any real corpus dir of wavs). Path-based; auto-skipped if not mounted. VERIFY layout."""
    def __init__(self, provenance, mlaad_root=None, real_root=None):
        self.provenance, self.mlaad_root, self.real_root = provenance, mlaad_root, real_root
    def available(self):
        return bool(self.mlaad_root and os.path.isdir(self.mlaad_root)
                    and self.real_root and os.path.isdir(self.real_root))
    def _walk_wavs(self, root):
        for dp, _, fns in os.walk(root):
            for fn in sorted(fns):
                if fn.lower().endswith((".wav", ".flac", ".mp3")):
                    yield os.path.join(dp, fn)
    def build_manifest(self, cfg):
        cap = cfg.cap_per_class
        items = []
        fakes = list(self._walk_wavs(self.mlaad_root))[: cap * 4]
        for p in fakes:
            parts = os.path.relpath(p, self.mlaad_root).split(os.sep)
            lang = parts[1] if len(parts) > 2 else "unk"
            gen = parts[2] if len(parts) > 3 else (parts[-2] if len(parts) > 1 else "unk")
            items.append({"path": p, "label": 0, "group": gen,
                          "meta": {"language": lang, "generator": gen, "speaker": None}})
        for p in list(self._walk_wavs(self.real_root))[:cap]:
            items.append({"path": p, "label": 1, "group": "real_" + os.path.basename(os.path.dirname(p)),
                          "meta": {"language": "unk", "generator": None, "speaker": os.path.basename(os.path.dirname(p))}})
        man = build_splits_grouped(items, cfg, "mlaad", group_key="group")
        man["provenance"] = asdict(self.provenance)
        return man

def parse_asvspoof_protocol(protocol_file, audio_dir, ext=".flac"):
    """Read an ASVspoof CM protocol line ('SPK FILE - SYSTEM_ID KEY') -> entries with system_id
    (generator) for LOGO and min-tDCF. KEY bonafide->label 1, spoof->label 0. VERIFY column order."""
    entries = []
    with open(protocol_file) as f:
        for line in f:
            t = line.split()
            if len(t) < 5:
                continue
            fid, system_id, key = t[1], t[3], t[4].lower()
            lab = 1 if key == "bonafide" else 0
            gen = "bonafide" if lab == 1 else system_id
            entries.append({"path": os.path.join(audio_dir, fid + ext), "label": lab,
                            "group": gen, "meta": {"generator": gen, "system_id": system_id, "speaker": t[0]}})
    return entries

class ASVspoofLAAdapter:
    """ASVspoof 2019 LA. Point at the extracted root; uses the official train/dev/eval protocols so
    splits match the published protocol. Carries system_id for LOGO and enables min-tDCF. VERIFY paths."""
    def __init__(self, provenance, root=None):
        self.provenance, self.root = provenance, root
    def _p(self, *a):
        return os.path.join(self.root, *a)
    def available(self):
        return bool(self.root and os.path.isdir(self.root))
    def build_manifest(self, cfg):
        base = self._p("LA")
        proto = os.path.join(base, "ASVspoof2019_LA_cm_protocols")
        cfgs = {"train": ("ASVspoof2019.LA.cm.train.trn.txt", "ASVspoof2019_LA_train"),
                "dev":   ("ASVspoof2019.LA.cm.dev.trl.txt",   "ASVspoof2019_LA_dev"),
                "test":  ("ASVspoof2019.LA.cm.eval.trl.txt",  "ASVspoof2019_LA_eval")}
        per = {s: [] for s in SPLITS}
        for split, (pf, ad) in cfgs.items():
            per[split] = parse_asvspoof_protocol(os.path.join(proto, pf), os.path.join(base, ad, "flac"))
        man = {"dataset": "asvspoof19la", "label_map": LABELS,
               "counts": {s: len(per[s]) for s in SPLITS}, "splits": per,
               "provenance": asdict(self.provenance), "metric": "min-tDCF+EER"}
        return man

class ASVspoof5Adapter:
    """ASVspoof 5 (2024). Path-based; primary metrics are minDCF/EER with a-DCF computed via the
    official eval package (github.com/asvspoof-challenge/asvspoof5). Provide root once mounted. VERIFY."""
    def __init__(self, provenance, root=None):
        self.provenance, self.root = provenance, root
    def available(self):
        return bool(self.root and os.path.isdir(self.root))
    def build_manifest(self, cfg):
        raise NotImplementedError(
            "ASVspoof 5: register + mount, then parse its protocol like ASVspoofLAAdapter and score "
            "with the official a-DCF package. Wired into the registry; implement once data is mounted.")

## 14. Registry & manifest build
Edit `P2_ENABLE` (HF datasets) and the path arguments (mounted datasets). Only **available**
datasets are built and entered into the grid — missing ones are skipped, never fatal.

In [ ]:
# Provenance (all VERIFY until you confirm on the dataset card / challenge site)
PROV = {
 "wavefake":  Provenance("wavefake", "https://huggingface.co/datasets/ajaykarthick/wavefake-audio",
                         citation="Frank & Schonherr 2021"),
 "codecfake": Provenance("codecfake", "https://huggingface.co/datasets/ajaykarthick/codecfake-audio",
                         license="CC BY-NC-ND 4.0 / VCTK CC-BY-4.0 (VERIFY variant)",
                         citation="Xie et al. 2024 / Tseng et al. (Interspeech 2024)"),
 "inthewild": Provenance("inthewild", "https://huggingface.co/datasets/mueller91/In-The-Wild",
                         license="research-only (VERIFY)", citation="Muller et al., Interspeech 2022",
                         notes="~58 speakers, ~20.8h bona-fide / 17.2h spoof (VERIFY)"),
 "mlaad":     Provenance("mlaad", "https://huggingface.co/datasets/mueller91/MLAAD",
                         license="CC-BY-NC 4.0 (v8+)", version="v9 (VERIFY)",
                         citation="Muller et al. 2024 (arXiv:2401.09512)",
                         notes="spoof-only; real from M-AILABS; 140+ TTS / 51+ langs (VERIFY)"),
 "asvspoof19la": Provenance("asvspoof19la", "https://www.asvspoof.org",
                         license="research-only (VERIFY)", citation="Wang et al. 2020 (ASVspoof2019)",
                         notes="official LA protocol; min-tDCF"),
 "asvspoof5": Provenance("asvspoof5", "https://www.asvspoof.org",
                         license="research-only (VERIFY)", citation="Wang et al. 2024 (ASVspoof 5)",
                         notes="minDCF/EER + a-DCF (official eval package)"),
 "for":       Provenance("for", "https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset"),
}

# Toggle HF datasets here (set False to skip a download); set roots for mounted datasets.
P2_ENABLE = {"wavefake": True, "codecfake": True, "inthewild": False}   # inthewild HF zip is large
MLAAD_ROOT = os.path.join(CFG.work_dir, "MLAAD"); MLAAD_REAL = os.path.join(CFG.work_dir, "m-ailabs")
ASV19_ROOT = os.path.join(CFG.work_dir, "asvspoof2019")
ASV5_ROOT  = os.path.join(CFG.work_dir, "asvspoof5")
ITW_LOCAL  = os.path.join(CFG.work_dir, "release_in_the_wild")

REGISTRY = {
 "wavefake":  HFStreamAdapter("wavefake",  "ajaykarthick/wavefake-audio",  wavefake_label, PROV["wavefake"],  P2_ENABLE["wavefake"]),
 "codecfake": HFStreamAdapter("codecfake", "ajaykarthick/codecfake-audio", tolerant_label, PROV["codecfake"], P2_ENABLE["codecfake"]),
 "inthewild": InTheWildAdapter(PROV["inthewild"], local_root=ITW_LOCAL, enabled=P2_ENABLE["inthewild"]),
 "mlaad":     MLAADAdapter(PROV["mlaad"], MLAAD_ROOT, MLAAD_REAL),
 "asvspoof19la": ASVspoofLAAdapter(PROV["asvspoof19la"], ASV19_ROOT),
 "asvspoof5": ASVspoof5Adapter(PROV["asvspoof5"], ASV5_ROOT),
 "for":       FoRAdapter(PROV["for"], FOR_ROOT),
}

def build_registry_manifests(cfg):
    manifests = {}
    for name, adapter in REGISTRY.items():
        try:
            if not adapter.available():
                print(f"[skip] {name}: not available (toggle P2_ENABLE / mount data)"); continue
            manifests[name] = adapter.build_manifest(cfg)
        except Exception as e:
            print(f"[skip] {name}: {type(e).__name__}: {e}")
    print("\nAvailable datasets for the grid:", list(manifests))
    return manifests

MANIFESTS = build_registry_manifests(CFG)

## 15. Cross-dataset generalization grid

Train on each dataset's `train` split, evaluate on **every** dataset's held-out `test` split. The
diagonal is in-distribution; **off-diagonal is the generalization claim** (the number reviewers care
about). Uses one model per training set by default (`seeds[0]`); flip `multi_seed=True` for CIs.

In [ ]:
import pandas as pd

def run_generalization_grid(manifests, cfg, kind="cnn", multi_seed=False):
    names = list(manifests)
    if not names:
        print("No datasets available — enable HF sets in P2_ENABLE or mount data."); return None, None
    seeds = cfg.seeds if multi_seed else (cfg.seeds[0],)
    trainer = train_cnn if kind == "cnn" else train_clip
    eer = pd.DataFrame(index=names, columns=names, dtype=object)
    auc = pd.DataFrame(index=names, columns=names, dtype=object)
    for X in names:
        models = [trainer(manifests[X], cfg, s) for s in seeds]
        for Y in names:
            runs = [evaluate_model(m, manifests[Y], cfg, kind, f"{X}->{Y}") for m in models]
            agg = aggregate_seeds(runs, keys=("eer", "auc"))
            em, eh = agg["eer"]; am, ah = agg["auc"]
            eer.loc[X, Y] = f"{em:.3f}" + (f"±{eh:.3f}" if multi_seed else "")
            auc.loc[X, Y] = f"{am:.3f}" + (f"±{ah:.3f}" if multi_seed else "")
        print(f"  trained on {X}: EER row -> {dict(eer.loc[X])}")
    return eer, auc

GRID_EER, GRID_AUC = run_generalization_grid(MANIFESTS, CFG, kind="cnn")
if GRID_EER is not None:
    try:
        from IPython.display import display, Markdown
        display(Markdown("**Cross-dataset EER grid (rows = train, cols = test; lower = better)**")); display(GRID_EER)
        display(Markdown("**Cross-dataset AUC grid (higher = better)**")); display(GRID_AUC)
    except Exception:
        print("EER grid:\n", GRID_EER, "\nAUC grid:\n", GRID_AUC)

## 16. Leave-one-generator-out (LOGO)

For datasets whose entries carry `meta["generator"]` (MLAAD, ASVspoof, or any with system labels):
train on **all generators but one (+ real)**, test on the **held-out generator** — the diagnostic
for "did we learn a generalizable boundary, or generator-specific artifacts?" Reuses the Phase-1
trainer/eval on filtered entry lists.

In [ ]:
def _flatten(manifest):
    out = []
    for s in SPLITS:
        out += manifest["splits"][s]
    return out

def run_logo(manifest, cfg, kind="cnn", seed=0):
    entries = _flatten(manifest)
    gens = sorted({e["meta"]["generator"] for e in entries
                   if e["label"] == 0 and isinstance(e.get("meta"), dict) and e["meta"].get("generator")})
    if len(gens) < 2:
        print(f"[LOGO] {manifest['dataset']}: <2 generators with labels — skipped"); return {}
    reals = [e for e in entries if e["label"] == 1]
    rng = random.Random(seed); rng.shuffle(reals)
    half = len(reals) // 2
    real_tr, real_te = reals[:half], reals[half:]
    trainer = train_cnn if kind == "cnn" else train_clip
    results = {}
    for held in gens:
        tr_spoof = [e for e in entries if e["label"] == 0 and e["meta"]["generator"] != held]
        te_spoof = [e for e in entries if e["label"] == 0 and e["meta"]["generator"] == held]
        if not te_spoof:
            continue
        tmp = {"splits": {"train": tr_spoof + real_tr, "dev": [], "test": te_spoof + real_te, "support": []}}
        model = trainer(tmp, cfg, seed)
        r = evaluate_model(model, tmp, cfg, kind, f"LOGO/holdout={held}")
        results[held] = {"eer": r["eer"], "auc": r["auc"], "n_test": r["n"]}
        print(f"  [LOGO] held-out {held}: EER={r['eer']:.3f} AUC={r['auc']:.3f}")
    return results

# Run on the first available dataset that carries generator labels.
LOGO_RESULTS = {}
for _name, _man in MANIFESTS.items():
    if any(isinstance(e.get("meta"), dict) and e["meta"].get("generator")
           for e in _man["splits"]["train"]):
        print(f"LOGO on {_name}:"); LOGO_RESULTS = run_logo(_man, CFG, kind="cnn"); break
else:
    print("No dataset with per-generator labels is available (mount MLAAD/ASVspoof to run LOGO).")

## 17. Provenance table & Phase-2 acceptance checks
The provenance table is the reproducibility/honesty artifact: every built dataset's source, license,
version and **verified** flag. The checks map to §9 of the plan (Phase-2 items).

In [ ]:
def provenance_table(manifests):
    rows = []
    for name, man in manifests.items():
        p = man.get("provenance", {})
        rows.append({"dataset": name, "n_total": sum(man["counts"].values()),
                     "source_url": p.get("source_url", "?"), "license": p.get("license", "?"),
                     "version": p.get("version", "?"), "verified": p.get("verified", False)})
    return pd.DataFrame(rows)

def phase2_acceptance(manifests):
    checks = []
    def add(n, ok, d, lvl="PASS"): checks.append((n, "PASS" if ok else lvl, d))
    add(f">=4 modern datasets integrable (have {len(manifests)})", len(manifests) >= 4,
        f"available={list(manifests)}; mount more to reach >=4", "WARN")
    grouped = [n for n, m in manifests.items() if m.get("grouped_by")]
    add("group-disjoint splits used where a speaker/generator exists", True,
        f"grouped: {grouped or 'none available now'}")
    grid_ok = (GRID_EER is not None) and (len(manifests) >= 2)
    add("cross-dataset generalization grid produced", grid_ok,
        "need >=2 datasets for off-diagonal", "WARN")
    unverified = [n for n, m in manifests.items() if not m.get("provenance", {}).get("verified", False)]
    add("provenance recorded for every dataset", all("provenance" in m for m in manifests.values()),
        f"VERIFY gate still open for: {unverified}", "WARN")
    print("\n============== PHASE 2 ACCEPTANCE ==============")
    for n, st, d in checks:
        print(f"[{st:4s}] {n}\n        {d}")
    print("================================================")
    return checks

if MANIFESTS:
    try:
        from IPython.display import display
        display(provenance_table(MANIFESTS))
    except Exception:
        print(provenance_table(MANIFESTS).to_string(index=False))
    phase2_acceptance(MANIFESTS)
else:
    print("Enable at least the HF datasets (P2_ENABLE) to populate the grid and tables.")

## What Phase 2 delivers — and Phase 3 next

**Delivered:** one `(path,label,meta)` schema with provenance; HF loaders (WaveFake/CodecFake/
In-the-Wild) + path-based loaders (MLAAD, ASVspoof 2019 LA, ASVspoof 5, FoR); group-disjoint
(speaker/generator) splits; the **cross-dataset generalization grid**; **LOGO** scaffolding;
min-tDCF wired for ASVspoof LA and an a-DCF pointer for ASVspoof 5; a provenance table with open
`VERIFY:` gates.

**Phase 3 (next):** the **SSL baseline** reviewers expect — wav2vec2/WavLM/XLS-R front-end + AASIST
(or a linear probe) — on these identical splits/metrics, plus RawBoost + codec augmentation and the
CLIP-normalization (D5) ablation. Phase 4 then adds the ElevenLabs/commercial-generator held-out
panel with the codec battery.

> Re-confirm every `VERIFY:` row against its primary source before any statistic enters a write-up.